# Day 10
## Part 1

Each line in the input text is a discrete _machine_, such as:

```
[.##.] (3) (1,3) (2) (2,3) (0,2) (0,1) {3,5,4,7}
```

Each machine has a set of lights which are either on `#` or off `.`, a set of button wirings, and the respective "joltage" requirements for the wirings. The goal is to press buttons such as to meet the lighting requirement (or, as we will use, to turn the lighting requirement to be all off):

In [2]:
def ints(string: str, sep: str = ','):
    return list(map(int, string.split(sep)))

machines: list[tuple[list[bool], list[list[bool]], list[int]]] = []

with open('10.txt') as f:
    for line in f.readlines():
        line = line.strip().removeprefix('[').removesuffix('}')
        lights, rest = line.split('] (')
        rest, joltage = rest.split(') {')

        lights = [c == '#' for c in lights]
        buttons = [ints(b) for b in rest.split(') (')]
        buttons = [[i in b for i in range(len(lights))] for b in buttons]

        joltage = ints(joltage)
        machines.append((lights, buttons, joltage))

def preview(wire: list[bool]):
    return ''.join('.#'[c] for c in wire)

# for l, bs, j in machines:
#     print(preview(l), [preview(b) for b in bs], j)

Note that `a ^ X ^ X = a` : using the same button twice will toggle the same lights, reverting to as if they were pressed once. Therefore at maximum each button will need to be pressed only once, and as such we can iterate through all possible on/off combinations:

In [3]:
def xor(a: list[bool], b: list[bool]):
    "XOR two lists of bools."
    return [ax ^ bx for ax, bx in zip(a, b)]

from itertools import product

def solve_p1(
    lights: list[bool],
    buttons: list[list[bool]]
):
    for presses in product([False, True], repeat=len(buttons)):
        l = lights
        cost = 0
        for p, b in zip(presses, buttons):
            if p:
                l = xor(l, b)
                cost += 1
        if not any(l):
            yield cost


For part 1, we ignore all costs and take only the count of presses:

In [4]:
costs = [
    min(solve_p1(lights, buttons))
    for lights, buttons, _ in machines
]
print(costs)
print(sum(costs))

[2, 3, 2]
7


# Part 2: Numeric increases
Now each button is not reversible: pressing it increases the corresponding 'counter' by 1 (from a start of 0). To take our first example,

```
(3) (1,3) (2) (2,3) (0,2) (0,1) {3,5,4,7}
```

we want some $x$ such that

$$

\begin{bmatrix}
0 & 0 & 0 & 1 & 0 & 0 \\
0 & 1 & 0 & 1 & 0 & 0 \\
0 & 1 & 0 & 0 & 0 & 0 \\
0 & 1 & 1 & 0 & 0 & 0 \\
1 & 0 & 1 & 0 & 0 & 0 \\
1 & 1 & 0 & 0 & 0 & 0
\end{bmatrix}
x =
\begin{bmatrix}
3 \\ 5 \\ 4 \\ 7 \\ 0 \\ 0
\end{bmatrix}
$$
Note that we expanded the matrices such that the central is square and thus invertible.


In [58]:
import numpy as np 

def square_pad(a, fill=0):
    "Pad an array such that it is square."
    n = max(a.shape)
    pads = tuple(
        (0, n-x) for x in a.shape)
    return np.pad(a, pad_width=pads, mode='constant', constant_values=0)

_, buttons, reqs = machines[0]
a = square_pad(np.array(buttons).astype(int))
n = max(a.shape)
b = reqs
b += [0] * (n-len(b))
b = np.array(b)
print(a)
print(b)

x, *_ = np.linalg.lstsq(a, b, rcond=None)
print(x)
print(a@x - b)

[[0 0 0 1 0 0]
 [0 1 0 1 0 0]
 [0 0 1 0 0 0]
 [0 0 1 1 0 0]
 [1 0 1 0 0 0]
 [1 1 0 0 0 0]]
[3 5 4 7 0 0]
[-3.27272727  2.72727273  3.81818182  2.81818182  0.          0.        ]
[-0.18181818  0.54545455 -0.18181818 -0.36363636  0.54545455 -0.54545455]


Unfortunately this doesn't work. As you can see by the lack of this being complete, I don't care enough to solve linear equations for integers.